In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage.filters import gaussian_filter1d


# Function to apply Gaussian smoothing
def smooth_gaussian(x, y):
    y_smoothed = gaussian_filter1d(y, sigma=1)
    return x, y_smoothed

In [ ]:
def plot_for_model(rho):

    df = pd.read_csv("project.csv")
    df = df.dropna()
    df = df[df["data.rho_name"] == rho]
    df["data.rho_param"] = df["data.rho_param"].apply(lambda x: eval(x)["alpha"])
    df["data.rho_param"] = df["data.rho_param"].astype(float)

    grouped = (
        df.groupby(["model", "data.rho_param"])
        .agg({"train/loss_epoch": "min", "val/loss": "min"})
        .reset_index()
    )
    # Splitting the dataframe based on unique tags for color distinction in plot
    model_names = grouped["model"].unique()
    plt.figure(figsize=(14, 6))

    # Subplot 1: Train loss vs. Delta with log scale on y-axis and Gaussian smoothing
    plt.subplot(1, 2, 1)
    for model in model_names:
        alpha_max = grouped[grouped["model"] == model]["data.rho_param"].max()
        alpha_min = grouped[grouped["model"] == model]["data.rho_param"].min()
        grouped.loc[grouped["model"] == model, "data.rho_param"] -= alpha_min
        grouped.loc[grouped["model"] == model, "data.rho_param"] /= alpha_max - alpha_min

        subset = grouped[grouped["model"] == model].sort_values(by="data.rho_param")
        # x_smooth, y_smooth = subset["data.rho_param"].values, subset["train/loss_epoch"].values
        x_smooth, y_smooth = smooth_gaussian(
            subset["data.rho_param"].values, subset["train/loss_epoch"].values
        )
        plt.plot(x_smooth, y_smooth, label=model)
        # plt.scatter(x_smooth, y_smooth, label=tag, s=1)
    plt.xlabel("alpha: complexity parameter")
    plt.ylabel("Mean Train Loss")
    plt.title(rho)
    plt.yscale("log")
    plt.legend(title="model")
    plt.show()

In [ ]:
plot_for_model("impulse")

In [ ]:
plot_for_model("exponential")

In [ ]:
plot_for_model("polynomial")

In [ ]:
plot_for_model("airy")

In [ ]:
dd